![TecNM](assets/encabezado.png)

---

# Machine Learning y Deep Learning
## Unidad 2 · Modelos de Predicción Supervisados

Práctica 3 — Árboles de Regresión

> **Facilitador:** Dr. José Gabriel Rodríguez Rivas  
> **Alumno:** Christian Gibran Espituñal Villanueva

---

## Objetivo

Construir, evaluar y optimizar un **árbol de regresión** para predecir el precio de automóviles a partir de variables técnicas, comparando distintas configuraciones de hiperparámetros y analizando el comportamiento del modelo ante el sobreajuste.

---

## Marco Teórico

Un **árbol de regresión** es un modelo de aprendizaje supervisado no paramétrico que divide el espacio de características en regiones rectangulares y predice el valor promedio de la variable objetivo en cada región (hoja).

La división en cada nodo se elige minimizando el error cuadrático medio ponderado de los subconjuntos resultantes:

$$\text{MSE}_{\text{split}} = \frac{n_L}{n} \cdot \text{MSE}_L + \frac{n_R}{n} \cdot \text{MSE}_R$$

| Característica | Regresión Lineal | Árbol de Regresión |
|---|---|---|
| Supuesto de linealidad | Sí | No |
| Captura interacciones | Limitado | Sí |
| Interpretabilidad | Alta | Media-Alta |
| Riesgo de sobreajuste | Bajo | Alto (sin poda) |
| Sensibilidad a outliers | Alta | Moderada |

---

## Nota sobre GPU

scikit-learn ejecuta los árboles de decisión en **CPU**. PyTorch se utiliza en este notebook exclusivamente para detectar y reportar las propiedades del dispositivo CUDA disponible.

---

## Conjunto de Datos

Archivo `datasets/autos2.csv` — especificaciones técnicas y precios de mercado de automóviles.

| Variable | Tipo | Descripción |
|---|---|---|
| `horsepower` | Predictora | Potencia del motor (cv) |
| `engine-size` | Predictora | Cilindrada del motor (cm³) |
| `city-mpg` | Predictora | Consumo urbano (millas/galón) |
| `wheel-base` | Predictora | Distancia entre ejes (pulgadas) |
| `bore` | Predictora | Diámetro del cilindro (pulgadas) |
| `price` | Objetivo | Precio del vehículo (USD) |

---

## Contenido

1. [Entorno y librerías](#1.-Entorno-y-librerías)
2. [Detección de dispositivo (PyTorch)](#2.-Detección-de-dispositivo)
3. [Carga y exploración inicial](#3.-Carga-y-exploración-inicial)
4. [Preprocesamiento](#4.-Preprocesamiento)
5. [División de datos](#5.-División-de-datos)
6. [Modelo base (max_depth=4)](#6.-Modelo-base)
7. [Visualización del árbol](#7.-Visualización-del-árbol)
8. [Comparación de hiperparámetros](#8.-Comparación-de-hiperparámetros)
9. [Análisis de sobreajuste](#9.-Análisis-de-sobreajuste)
10. [Modelo optimizado](#10.-Modelo-optimizado)
11. [Importancia de variables](#11.-Importancia-de-variables)
12. [Conclusiones](#12.-Conclusiones)

---
## 1. Entorno y librerías

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

# PyTorch — solo para detección e información del dispositivo CUDA
import torch

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.28,
    'axes.labelsize': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'font.family': 'DejaVu Sans',
})

C_BLUE, C_ORANGE, C_GREEN, C_RED, C_PURPLE = (
    '#005F9E', '#E87722', '#3DAD6B', '#C0392B', '#8E44AD'
)
PALETTE = [C_BLUE, C_ORANGE, C_GREEN, C_RED, C_PURPLE]
sns.set_theme(style='whitegrid', palette=PALETTE)

RANDOM_STATE = 42
FEATURES     = ['horsepower', 'engine-size', 'city-mpg', 'wheel-base', 'bore']
TARGET       = 'price'
DATASET_PATH = 'datasets/autos2.csv'

print('Entorno configurado correctamente.')

---
## 2. Detección de dispositivo (PyTorch)

PyTorch se usa exclusivamente para detectar y reportar el hardware CUDA disponible. El entrenamiento del árbol de decisión ocurre en CPU vía scikit-learn.

In [ ]:
CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE         = torch.device('cuda' if CUDA_AVAILABLE else 'cpu')

rows = [
    ('PyTorch',          torch.__version__),
    ('CUDA disponible',  str(CUDA_AVAILABLE)),
    ('Dispositivo CUDA', str(DEVICE)),
]

if CUDA_AVAILABLE:
    props = torch.cuda.get_device_properties(0)
    rows += [
        ('GPU',          torch.cuda.get_device_name(0)),
        ('Compute cap.', f'sm_{props.major}{props.minor}  ({props.multi_processor_count} SMs)'),
        ('VRAM total',   f'{props.total_memory / 1024**3:.2f} GB'),
        ('CUDA version', torch.version.cuda),
    ]

rows += [
    ('Arbol backend',   'sklearn.DecisionTreeRegressor'),
    ('Arbol ejecucion', 'CPU'),
]

display(
    pd.DataFrame(rows, columns=['Parametro', 'Valor'])
    .style.hide(axis='index')
    .set_caption('Configuracion del entorno de computo')
)

---
## 3. Carga y exploración inicial

In [ ]:
df = pd.read_csv(DATASET_PATH)

print(f'Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
display(df.head())

In [ ]:
display(
    df[FEATURES + [TARGET]]
    .describe().T
    .style
    .format(precision=2)
    .background_gradient(cmap='Blues', subset=['mean', 'std'])
    .set_caption('Estadisticas descriptivas')
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Exploracion de la variable objetivo', fontsize=14, fontweight='bold', y=1.01)

sns.histplot(df[TARGET].dropna(), bins=30, kde=True,
             color=C_BLUE, edgecolor='white', ax=axes[0])
axes[0].xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].set_title('Distribucion del precio')
axes[0].set_xlabel('Precio (USD)')
axes[0].set_ylabel('Frecuencia')

sns.boxplot(x=df[TARGET].dropna(), color=C_BLUE, linewidth=1.5,
            flierprops=dict(marker='o', alpha=0.5), ax=axes[1])
axes[1].xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].set_title('Diagrama de cajas y bigotes — Precio')
axes[1].set_xlabel('Precio (USD)')

plt.tight_layout()
plt.show()

In [ ]:
corr = df[FEATURES + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
    annot_kws={'size': 9}
)
ax.set_title('Matriz de correlacion')
plt.tight_layout()
plt.show()

---
## 4. Preprocesamiento

In [ ]:
data = df[FEATURES + [TARGET]].dropna().reset_index(drop=True)

print(f'Registros antes de limpiar  : {len(df)}')
print(f'Registros despues de limpiar: {len(data)}')
print(f'Valores nulos restantes     : {data.isnull().sum().sum()}')

---
## 5. División de datos

In [ ]:
X = data[FEATURES]
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print(f'Total         : {len(data)}')
print(f'Entrenamiento : {X_train.shape[0]} muestras ({X_train.shape[0]/len(data):.0%})')
print(f'Prueba        : {X_test.shape[0]}  muestras ({X_test.shape[0]/len(data):.0%})')

---
## 6. Modelo base

In [ ]:
def evaluar_modelo(model, X_tr, y_tr, X_te, y_te, nombre='Modelo'):
    """Entrena, evalua y devuelve un dict con metricas."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    mse    = mean_squared_error(y_te, y_pred)
    mae    = mean_absolute_error(y_te, y_pred)
    rmse   = np.sqrt(mse)
    r2     = r2_score(y_te, y_pred)
    cv_r2  = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2').mean()

    print(f'  {nombre}')
    print(f'  {"─"*46}')
    print(f'  MSE            : {mse:>12,.2f} USD2')
    print(f'  RMSE           : ${rmse:>10,.2f} USD')
    print(f'  MAE            : ${mae:>10,.2f} USD')
    print(f'  R2             : {r2:>12.4f}')
    print(f'  CV-R2 (5-fold) : {cv_r2:>12.4f}')
    print()
    return {'nombre': nombre, 'mse': mse, 'rmse': rmse,
            'mae': mae, 'r2': r2, 'cv_r2': cv_r2,
            'y_pred': y_pred, 'model': model}


def graficar_evaluacion(y_test, y_pred, titulo_sufijo=''):
    """Panel de tres graficas: KDE, scatter real vs predicho, residuos."""
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    fig.suptitle(f'Evaluacion del modelo — {titulo_sufijo}',
                 fontsize=13, fontweight='bold', y=1.01)
    fmt = mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k')

    # 1. KDE real vs predicho
    sns.kdeplot(y_test,  label='Real',     color=C_BLUE,   linewidth=2.2, ax=axes[0])
    sns.kdeplot(y_pred,  label='Predicho', color=C_ORANGE,
                linewidth=2.2, linestyle='--', ax=axes[0])
    axes[0].xaxis.set_major_formatter(fmt)
    axes[0].set_title('Distribucion de densidad')
    axes[0].set_xlabel('Precio (USD)')
    axes[0].set_ylabel('Densidad')
    axes[0].legend(frameon=True)

    # 2. Scatter real vs predicho
    lim_min = min(y_test.min(), y_pred.min()) * 0.90
    lim_max = max(y_test.max(), y_pred.max()) * 1.08
    axes[1].scatter(y_test, y_pred, color=C_BLUE, alpha=0.70,
                    edgecolors='white', linewidths=0.4, s=60)
    axes[1].plot([lim_min, lim_max], [lim_min, lim_max],
                 '--', color=C_RED, linewidth=1.8, label='Prediccion perfecta')
    axes[1].fill_between([lim_min, lim_max],
                         [lim_min, lim_max], [lim_min, lim_max * 1.5],
                         alpha=0.04, color=C_GREEN)
    axes[1].fill_between([lim_min, lim_max],
                         [lim_min, lim_max * 1.5], [lim_min, lim_max],
                         alpha=0.04, color=C_RED)
    axes[1].xaxis.set_major_formatter(fmt)
    axes[1].yaxis.set_major_formatter(fmt)
    axes[1].set_xlim(lim_min, lim_max)
    axes[1].set_ylim(lim_min, lim_max)
    axes[1].set_title('Real vs. Predicho')
    axes[1].set_xlabel('Precio real (USD)')
    axes[1].set_ylabel('Precio predicho (USD)')
    axes[1].legend(fontsize=9, frameon=True)
    axes[1].text(0.05, 0.92, f'R2 = {r2_score(y_test, y_pred):.4f}',
                 transform=axes[1].transAxes, fontsize=10, color='#333')

    # 3. Distribucion de residuos
    residuos = np.asarray(y_test) - y_pred
    sns.histplot(residuos, bins=25, kde=True, color=C_GREEN,
                 edgecolor='white', ax=axes[2])
    axes[2].axvline(0, color=C_RED, linestyle='--', linewidth=1.5)
    axes[2].xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
    axes[2].set_title('Distribucion de residuos')
    axes[2].set_xlabel('Residuo (Real - Predicho)')
    axes[2].set_ylabel('Frecuencia')
    axes[2].text(0.97, 0.95,
                 f'Media : ${residuos.mean():,.0f}\nDesv. : ${residuos.std():,.0f}',
                 transform=axes[2].transAxes, fontsize=8.5,
                 va='top', ha='right', color='#333')

    plt.tight_layout()
    plt.show()


tree_base = DecisionTreeRegressor(max_depth=4, random_state=RANDOM_STATE)
res_base  = evaluar_modelo(
    tree_base, X_train, y_train, X_test, y_test,
    'Arbol base (max_depth=4)'
)

graficar_evaluacion(y_test, res_base['y_pred'], 'Arbol base (max_depth=4)')

---
## 7. Visualización del árbol

In [ ]:
fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(
    res_base['model'],
    feature_names=X.columns.tolist(),
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
    impurity=False,
    precision=1,
)
ax.set_title('Arbol de Decision — Prediccion del Precio (max_depth=4)',
             fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

print(export_text(res_base['model'], feature_names=FEATURES))

---
## 8. Comparación de hiperparámetros

In [ ]:
profundidades = range(1, 16)
resultados = []

for d in profundidades:
    m = DecisionTreeRegressor(max_depth=d, random_state=RANDOM_STATE)
    m.fit(X_train, y_train)
    resultados.append({
        'max_depth' : d,
        'R2 Train'  : r2_score(y_train, m.predict(X_train)),
        'R2 Test'   : r2_score(y_test,  m.predict(X_test)),
        'RMSE Test' : np.sqrt(mean_squared_error(y_test, m.predict(X_test))),
    })

df_res = pd.DataFrame(resultados)

display(
    df_res.style
    .format({'R2 Train': '{:.4f}', 'R2 Test': '{:.4f}', 'RMSE Test': '${:,.0f}'})
    .background_gradient(cmap='RdYlGn', subset=['R2 Test'])
    .highlight_max(subset=['R2 Test'],   color='#D4EFDF')
    .highlight_min(subset=['RMSE Test'], color='#D4EFDF')
    .hide(axis='index')
    .set_caption('R2 y RMSE por profundidad')
)

---
## 9. Análisis de sobreajuste

In [ ]:
mejor_depth = int(df_res.loc[df_res['R2 Test'].idxmax(), 'max_depth'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Analisis sesgo-varianza segun profundidad del arbol',
             fontsize=13, fontweight='bold')

# R2
axes[0].plot(df_res['max_depth'], df_res['R2 Train'], 'o-',
             color=C_BLUE,   linewidth=2, label='Entrenamiento')
axes[0].plot(df_res['max_depth'], df_res['R2 Test'],  's-',
             color=C_ORANGE, linewidth=2, label='Prueba')
axes[0].axvline(mejor_depth, color='gray', linestyle=':',
                alpha=0.7, label=f'mejor depth = {mejor_depth}')
axes[0].set_xlabel('max_depth')
axes[0].set_ylabel('R2')
axes[0].set_title('Coeficiente de determinacion R2')
axes[0].legend(frameon=True)
axes[0].set_xticks(list(profundidades))

# RMSE
axes[1].plot(df_res['max_depth'], df_res['RMSE Test'], 'D-',
             color=C_RED, linewidth=2, label='RMSE Prueba')
axes[1].axvline(int(df_res.loc[df_res['RMSE Test'].idxmin(), 'max_depth']),
                color='gray', linestyle=':', alpha=0.7)
axes[1].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].set_xlabel('max_depth')
axes[1].set_ylabel('RMSE (USD)')
axes[1].set_title('RMSE en conjunto de prueba')
axes[1].legend(frameon=True)
axes[1].set_xticks(list(profundidades))

plt.tight_layout()
plt.show()

print(f'Mejor profundidad segun R2 en prueba: max_depth = {mejor_depth}')

---
## 10. Modelo optimizado

In [ ]:
tree_opt = DecisionTreeRegressor(
    max_depth=7,
    min_samples_split=10,
    min_samples_leaf=2,
    max_features='sqrt',
    max_leaf_nodes=20,
    random_state=RANDOM_STATE,
)

res_opt = evaluar_modelo(
    tree_opt, X_train, y_train, X_test, y_test,
    'Arbol optimizado (depth=7 + restricciones)'
)

graficar_evaluacion(y_test, res_opt['y_pred'], 'Arbol optimizado')

In [ ]:
tabla = pd.DataFrame([
    {'Modelo'    : r['nombre'],
     'RMSE (USD)': r['rmse'],
     'MAE (USD)' : r['mae'],
     'R2 Test'   : r['r2'],
     'CV-R2'     : r['cv_r2']}
    for r in [res_base, res_opt]
])

display(
    tabla.style
    .format({'RMSE (USD)': '${:,.0f}', 'MAE (USD)': '${:,.0f}',
             'R2 Test': '{:.4f}', 'CV-R2': '{:.4f}'})
    .highlight_max(subset=['R2 Test', 'CV-R2'],         color='#D4EFDF')
    .highlight_min(subset=['RMSE (USD)', 'MAE (USD)'],  color='#D4EFDF')
    .hide(axis='index')
    .set_caption('Comparacion de modelos')
)

---
## 11. Importancia de variables

In [ ]:
importancias = pd.Series(
    res_opt['model'].feature_importances_, index=FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(importancias.index, importancias.values,
               color=C_BLUE, edgecolor='white', height=0.55)

for bar, val in zip(bars, importancias.values):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.1%}', va='center', fontsize=9)

ax.set_xlim(0, importancias.max() * 1.2)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_title('Importancia de caracteristicas\n(Modelo optimizado)')
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.show()

---
## 12. Conclusiones

| Aspecto | Hallazgo |
|---|---|
| **Variable mas influyente** | `engine-size` es el primer criterio de division en ambos modelos, siendo el predictor mas relevante del precio. |
| **Rendimiento base** | Con `max_depth=4`, el arbol alcanza R²=0.92 y RMSE≈$3,031, un buen punto de partida. |
| **Sobreajuste** | Profundidades > 9 generan sobreajuste: R² de entrenamiento tiende a 1.0 mientras el de prueba se deteriora. |
| **Modelo optimizado** | `max_depth=7` con restricciones adicionales (`min_samples_split`, `max_leaf_nodes`) reduce el RMSE y mejora la generalizacion (CV-R² mas alto). |
| **Limitacion principal** | Para vehiculos de precio > $30,000, el modelo tiende a subestimar el precio real, posiblemente por escasez de ejemplos en ese rango. |
| **Rol de PyTorch** | Utilizado exclusivamente para detectar y reportar el entorno CUDA. El entrenamiento ocurre integramente en CPU via scikit-learn. |